In [17]:
from notion_client import Client
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

NOTION_TOKEN = os.getenv("NOTION_TOKEN")
notion = Client(auth=NOTION_TOKEN, notion_version="2025-09-03")

output_dir = Path("extracted_notion_data")
output_dir.mkdir(exist_ok=True)

visited_nodes = set() # 중복 방문 방지

## 1. 보조 함수 정의

In [18]:
def get_safe_name(name):
    """윈도우 파일 시스템에 안전한 이름으로 변환합니다."""
    if not name:
        return "Untitled"
    # 금지 문자 제거
    name = re.sub(r'[\\/:*?"<>|]', '_', name)
    # 양끝 공백 및 마침표 제거 (윈도우 오류 방지)
    name = name.strip(" . ")
    return name if name else "Untitled"

def get_all_blocks(parent_id):
    blocks = []
    has_more = True
    start_cursor = None
    while has_more:
        try:
            response = notion.blocks.children.list(block_id=parent_id, start_cursor=start_cursor)
            blocks.extend(response["results"])
            has_more = response["has_more"]
            start_cursor = response["next_cursor"]
        except Exception as e:
            print(f"    [!] 블록 로드 실패 ({parent_id}): {e}")
            break
    return blocks

def get_page_title(page_obj):
    properties = page_obj.get("properties", {})
    for prop in properties.values():
        if prop["type"] == "title":
            titles = prop.get("title", [])
            if titles:
                return "".join([t["plain_text"] for t in titles])
    return "Untitled"

## 2. 재귀적 추출 엔진 (Error-Resistant)

In [19]:
def process_content_recursive(node_id, current_path, node_type="page"):
    if node_id in visited_nodes:
        return
    visited_nodes.add(node_id)
    
    try:
        if node_type == "page":
            # 1. 페이지 정보 가져오기
            try:
                page_obj = notion.pages.retrieve(page_id=node_id)
                title = get_page_title(page_obj)
            except:
                title = f"Page_{node_id[:8]}"
            
            safe_title = get_safe_name(title)
            print(f"📄 처리 중: {' > '.join(current_path.parts[1:])} > {safe_title}")
            
            # 2. 블록들 처리
            current_path.mkdir(parents=True, exist_ok=True)
            markdown_content = ""
            blocks = get_all_blocks(node_id)
            
            for block in blocks:
                b_type = block["type"]
                
                if b_type == "child_page":
                    c_title = block["child_page"]["title"]
                    markdown_content += f"\n> [하위 페이지: {c_title}](파일 저장됨)\n\n"
                    process_content_recursive(block["id"], current_path / safe_title, "page")
                
                elif b_type == "child_database":
                    c_title = block["child_database"]["title"]
                    markdown_content += f"\n> [하위 데이터베이스: {c_title}](폴더 저장됨)\n\n"
                    process_content_recursive(block["id"], current_path / safe_title / get_safe_name(c_title), "database")
                
                elif b_type in ["paragraph", "heading_1", "heading_2", "heading_3", "bulleted_list_item", "numbered_list_item"]:
                    rich_text = block[b_type].get("rich_text", [])
                    text = "".join([t["plain_text"] for t in rich_text])
                    if text:
                        if b_type == "heading_1": markdown_content += f"# {text}\n\n"
                        elif b_type == "heading_2": markdown_content += f"## {text}\n\n"
                        elif b_type == "heading_3": markdown_content += f"### {text}\n\n"
                        elif b_type == "bulleted_list_item": markdown_content += f"- {text}\n"
                        else: markdown_content += f"{text}\n\n"
            
            # 파일 저장
            with open(current_path / f"{safe_title}.md", "w", encoding="utf-8") as f:
                f.write(markdown_content)
                
        elif node_type == "database":
            print(f"🗃️ 데이터베이스 탐색: {' > '.join(current_path.parts[1:])}")
            current_path.mkdir(parents=True, exist_ok=True)
            try:
                db_pages = notion.databases.query(database_id=node_id).get("results", [])
                for page in db_pages:
                    process_content_recursive(page["id"], current_path, "page")
            except Exception as e:
                print(f"    [!] DB 쿼리 실패: {e}")
                
    except OSError as e:
        print(f"    [⚠️ 경로 오류 건너뜀] {current_path} / {node_id}: {e}")
    except Exception as e:
        print(f"    [❌ 일반 오류 건너뜀] {node_id}: {e}")

## 3. 실행 엔진

In [20]:
def start_recursive_crawl():
    print("🚀 안전한 재귀적 크롤링 시작...")
    try:
        results = notion.search().get("results", [])
        for item in results:
            process_content_recursive(item["id"], output_dir, node_type=item["object"])
        print("\n✅ 모든 데이터 추출 시도가 완료되었습니다!")
    except Exception as e:
        print(f"전체 탐색 중 오류 발생: {e}")

if NOTION_TOKEN:
    start_recursive_crawl()
else:
    print("NOTION_TOKEN이 설정되지 않았습니다.")

🚀 안전한 재귀적 크롤링 시작...
📄 처리 중:  > Getting started with meeting notes
📄 처리 중:  > important URL
📄 처리 중:  > 중간 발표 피드백 요약
📄 처리 중:  > 주요 문서
📄 처리 중: 주요 문서 > 회의록
📄 처리 중: 주요 문서 > 회의록 > 240319
📄 처리 중: 주요 문서 > 회의록 > 240321
📄 처리 중: 주요 문서 > 회의록 > 240325
📄 처리 중: 주요 문서 > 회의록 > 240401
📄 처리 중: 주요 문서 > 회의록 > 240402
📄 처리 중: 주요 문서 > 회의록 > 240403
📄 처리 중: 주요 문서 > 회의록 > 240406 - 백엔드 회의
📄 처리 중: 주요 문서 > 회의록 > 240419 _ 백엔드 미팅
📄 처리 중: 주요 문서 > 회의록 > 240503 - 백엔드 회의록
📄 처리 중: 주요 문서 > 발표 자료
📄 처리 중: 주요 문서 > 발표 자료 > 컨셉 발표
📄 처리 중: 주요 문서 > 트렐로
🗃️ 데이터베이스 탐색: 주요 문서 > 트렐로 > 사용자
    [!] DB 쿼리 실패: 'DatabasesEndpoint' object has no attribute 'query'
🗃️ 데이터베이스 탐색: 주요 문서 > 트렐로 > 모임 개최자
    [!] DB 쿼리 실패: 'DatabasesEndpoint' object has no attribute 'query'
🗃️ 데이터베이스 탐색: 주요 문서 > 트렐로 > 모든 사용자
    [!] DB 쿼리 실패: 'DatabasesEndpoint' object has no attribute 'query'
🗃️ 데이터베이스 탐색: 주요 문서 > 트렐로 > 광고주
    [!] DB 쿼리 실패: 'DatabasesEndpoint' object has no attribute 'query'
🗃️ 데이터베이스 탐색: 주요 문서 > 트렐로 > 광고주
    [!] DB 쿼리 실패: 'DatabasesEndpoint' obj